# Population density per municipality

This notebook maps the population density of all Dutch municipalities and saves it for the regression (notebook 7).

Population density is strongly right-skewed: a few cities are far denser than the rest of the country. The regression therefore uses the natural logarithm of population density. For the same reason the map uses a logarithmic colour scale; otherwise almost all municipalities would get the same light colour.

**Required files:**
- `data/raw/kwb2024_per_gemeente.csv`: population density per municipality (inhabitants per km² of land), extracted from CBS Kerncijfers wijken en buurten 2024 (85984NED)
- `data/raw/gemeenten_2024.gpkg`: municipal boundaries from CBS/PDOK, 2024 municipal division
- `data/processed/chargingpoints_households_income.csv`: created by notebook 1, only used to check that the names match

**Output:**
- `data/processed/population_density_per_municipality.csv`
- `figures/map_population_density_per_municipality.png`

In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

## 1. Load the data

In [ ]:
kwb = pd.read_csv("../data/raw/kwb2024_per_gemeente.csv")

density = kwb[["gemeentecode", "gemeente", "bevolkingsdichtheid"]].rename(columns={
    "gemeentecode": "municipality_code",
    "gemeente": "municipality",
    "bevolkingsdichtheid": "population_density",
})

# Natural logarithm (as used in the regression)
density["log_population_density"] = np.log(density["population_density"]).round(4)

print("Number of municipalities:", len(density))
print("Missing values:", density["population_density"].isna().sum())
display(density["population_density"].describe().round(0))

## 2. Distribution: why a logarithm?

The histogram on the left shows that most municipalities have fewer than 1,000 inhabitants per km², while a few cities have more than 5,000. After taking the logarithm (right) the distribution is much more symmetric.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(density["population_density"], bins=40, color="#7a5195", edgecolor="white")
axes[0].set_xlabel("Population density (inhabitants per km²)")
axes[0].set_ylabel("Number of municipalities")
axes[0].set_title("Population density")

axes[1].hist(density["log_population_density"], bins=40, color="#7a5195", edgecolor="white")
axes[1].set_xlabel("ln(population density)")
axes[1].set_ylabel("Number of municipalities")
axes[1].set_title("Log of population density")

plt.tight_layout()
plt.show()

## 3. Join the data to the map

Both files contain the CBS municipality code (for example `GM0014`), so the data can be joined on the code instead of the name.

In [ ]:
municipalities = gpd.read_file("../data/raw/gemeenten_2024.gpkg", layer="gemeente_gegeneraliseerd")

map_gdf = municipalities.merge(density, left_on="statcode", right_on="municipality_code", how="left")

# Check: should print 0
print("Municipalities without data:", map_gdf["population_density"].isna().sum())

## 4. Create the map

In [ ]:
fig, ax = plt.subplots(figsize=(10, 12))

map_gdf.plot(
    column="population_density",
    cmap="YlGnBu",
    norm=LogNorm(vmin=map_gdf["population_density"].min(), vmax=map_gdf["population_density"].max()),
    linewidth=0.3,
    edgecolor="white",
    legend=True,
    legend_kwds={"label": "Inhabitants per km² of land (log scale)", "shrink": 0.6},
    ax=ax,
    missing_kwds={"color": "lightgrey", "label": "No data"},
)

# Readable numbers on the colour bar instead of 10^2 and 10^3
colorbar = fig.axes[-1]
ticks = [25, 50, 100, 250, 500, 1000, 2500, 5000]
colorbar.minorticks_off()
colorbar.set_yticks(ticks)
colorbar.set_yticklabels([f"{t:,}" for t in ticks])

ax.set_title("Population density per municipality, 2024", fontsize=16)
ax.axis("off")

plt.tight_layout()
os.makedirs("../figures", exist_ok=True)
plt.savefig("../figures/map_population_density_per_municipality.png", dpi=200)
plt.show()

## 5. Save for the analysis

We use the same municipality names as the charging point file (the same mapping as in notebook 2), so all processed files can be merged on `municipality`.

In [ ]:
name_mapping = {
    "'s-Gravenhage": "Den Haag",
    "Beek (L.)": "Beek",
    "Hengelo (O.)": "Hengelo",
    "Laren (NH.)": "Laren",
    "Middelburg (Z.)": "Middelburg",
    "Rijswijk (ZH.)": "Rijswijk",
    "Stein (L.)": "Stein",
}
density_save = density[["municipality", "population_density", "log_population_density"]].copy()
density_save["municipality"] = density_save["municipality"].str.strip().replace(name_mapping)

# Check: does every municipality match the charging point data?
charging = pd.read_csv("../data/processed/chargingpoints_households_income.csv")
print("Only in the density data:", set(density_save["municipality"]) - set(charging["municipality"]))
print("Only in the charging data:", set(charging["municipality"]) - set(density_save["municipality"]))

density_save.to_csv("../data/processed/population_density_per_municipality.csv", index=False)
print("Saved:", len(density_save), "municipalities")